# Exploring Spectrogram

In [1]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from typing import List

In [2]:
AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 16, 19]
interval_width = 20  # ms

In [3]:
class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel - explicitly convert to single channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram with adjusted parameters
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                hop_length=int(sr * interval_width / 1000),
                n_mels=32
            )(waveform)

            # here, the mel_spec has the shape [1, n_mels=128, time_frames]
            # reshape it to [n_mels=128, time_frames]
            mel_spec = mel_spec.squeeze(0)

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Truncate to last break + 2
            last_break = labels.nonzero(as_tuple=True)[0][-1]
            mel_spec = mel_spec[:, :last_break+2]
            labels = labels[:last_break+2]
            
            print(f"Index: {idx}, Mel shape: {mel_spec.shape}, Labels shape: {labels.shape}")

            self.data.append((mel_spec, labels))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
dataset = BreakDataset()
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
)


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 16, Mel shape: torch.Size([32, 9842]), Labels shape: torch.Size([9842])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES  = 2  # because we have binary labels, duh

class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=128, num_layers=1):
        super(CNNRNN, self).__init__()
        cnn_output_size = 16
        self.conv = nn.Conv1d(n_mels, cnn_output_size, kernel_size=3, padding=1)
        self.rnn = nn.LSTM(input_size=cnn_output_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        # small sequence of linear
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        # x: (batch, n_mels, time_frames)
        x = F.relu(self.conv(x))  # (batch, cnn_output_size, time_frames)
        x = x.permute(0, 2, 1)  # (batch, time_frames, cnn_output_size)
        x, _ = self.rnn(x)  # (batch, time_frames, hidden_size)
        x = self.fc(x)  # (batch, time_frames, num_classes)
        x = x * mask.unsqueeze(-1)
        return x

# Initialize the model
model = CNNRNN()


In [88]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Initialize the dataset and data loader
dataset = BreakDataset(INDEX_SET, AUDIO_DIR, LABEL_DIR)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Initialize the model, optimizer, and loss function
model = CNNRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

# Train the model
epochs = 3
for epoch in range(epochs):
    for batch in dataloader:
        mel_spec, labels = batch
        max_length = mel_spec.shape[1]
        mel_specs = []
        labels_list = []
        for i in range(mel_spec.shape[0]):
            mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
            labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))
        mel_specs = torch.stack(mel_specs)
        labels = torch.stack(labels_list)
        mask = (mel_specs!= 0).any(dim=1).float()
        output = model(mel_specs, mask)
        loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

    # Save the model
    torch.save(model.state_dict(), f"saved_models/model_epoch_{epoch+1}.pth")


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 16, Mel shape: torch.Size([32, 9842]), Labels shape: torch.Size([9842])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])
Epoch 1, Loss: 0.6931471824645996
Epoch 2, Loss: 0.6931471824645996
Epoch 3, Loss: 0.6931471824645996
